# 03 | SQL na Prática

Carga em SQLite local e execução de consultas analíticas do projeto.

In [ ]:
import sqlite3
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
import pandas as pd
from src.config import PROCESSED_CLIENTES, PROCESSED_CATEGORIAS, PROCESSED_TRANSACOES


In [ ]:
schema = (ROOT / 'sql' / 'schema.sql').read_text(encoding='utf-8')
queries = (ROOT / 'sql' / 'queries_analiticas.sql').read_text(encoding='utf-8')
conn = sqlite3.connect(':memory:')
conn.executescript(schema)
pd.read_csv(PROCESSED_CLIENTES).to_sql('clientes', conn, if_exists='append', index=False)
pd.read_csv(PROCESSED_CATEGORIAS).to_sql('categorias', conn, if_exists='append', index=False)
pd.read_csv(PROCESSED_TRANSACOES).drop(columns=['ano_mes', 'flag_outlier']).to_sql('transacoes', conn, if_exists='append', index=False)


In [ ]:
query = '''
SELECT c.estado, COUNT(*) AS qtd, ROUND(AVG(t.valor), 2) AS ticket
FROM transacoes t JOIN clientes c ON c.cliente_id = t.cliente_id
GROUP BY c.estado ORDER BY qtd DESC;
'''
display(pd.read_sql_query(query, conn))
print(queries[:1500])
